# ЛР 5

## Импорты

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split

## Загрузка и предобработка

In [10]:
df = pd.read_csv('poems.csv', encoding='utf-8')
texts = df['text'].fillna('').tolist()

texts = texts[:5000]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яё\s\n]', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r' +', ' ', text).strip()
    return text

cleaned = [clean_text(t) for t in texts]

tokens = []
for line in cleaned:
    tokens.extend(line.split())

word_counts = Counter(tokens)
vocab_size = 5000
most_common = [w for w, _ in word_counts.most_common(vocab_size-1)]
vocab = most_common + ['<UNK>']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)
print("Размер словаря:", vocab_size)

def text_to_indices(text):
    return [word2idx.get(w, word2idx['<UNK>']) for w in text.split()]

sequences = []
targets = []
seq_len = 6
for text in cleaned:
    idx_seq = text_to_indices(text)
    if len(idx_seq) > seq_len:
        for i in range(len(idx_seq) - seq_len):
            sequences.append(idx_seq[i:i+seq_len])
            targets.append(idx_seq[i+seq_len])

X = torch.tensor(sequences, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

Размер словаря: 5000


## Модель

In [11]:
class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # берём последний выход
        return self.fc(out)

model = TextGenerator(vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

## Обучение

In [12]:
epochs = 20
for epoch in range(epochs):
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

Epoch 1/20, Loss: 5.0804
Epoch 2/20, Loss: 4.8368
Epoch 3/20, Loss: 4.7060
Epoch 4/20, Loss: 4.6062
Epoch 5/20, Loss: 4.5311
Epoch 6/20, Loss: 4.4730
Epoch 7/20, Loss: 4.4266
Epoch 8/20, Loss: 4.3893
Epoch 9/20, Loss: 4.3594
Epoch 10/20, Loss: 4.3328
Epoch 11/20, Loss: 4.3120
Epoch 12/20, Loss: 4.2943
Epoch 13/20, Loss: 4.2791
Epoch 14/20, Loss: 4.2671
Epoch 15/20, Loss: 4.2553
Epoch 16/20, Loss: 4.2458
Epoch 17/20, Loss: 4.2379
Epoch 18/20, Loss: 4.2295
Epoch 19/20, Loss: 4.2246
Epoch 20/20, Loss: 4.2190


## Генерация

In [13]:
def generate_text(seed, length=20, temperature=0.8):
    model.eval()
    words = seed.lower().split()
    for _ in range(length):
        seq = words[-seq_len:]
        seq_indices = [word2idx.get(w, word2idx['<UNK>']) for w in seq]
        x = torch.tensor([seq_indices], dtype=torch.long).to(device)
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits / temperature, dim=1).cpu().numpy()[0]
        next_idx = np.random.choice(len(probs), p=probs)
        next_word = idx2word[next_idx]
        words.append(next_word)
    return ' '.join(words)

print(generate_text("осень", length=15))

осень когда над ним и с каждым днем <UNK> и <UNK> <UNK> и <UNK> угрюмый той
